# 05 — MLflow Datasets: Tracking & Lineage

**UI tab:** Datasets

MLflow tracks **which dataset each run used** — including a hash of the data. When the data changes, the hash changes, giving you full lineage: you can always trace a metric back to the exact dataset version that produced it.

```
eval-questions (v1, hash: abc)   →  Run A  →  avg_score = 4.1
eval-questions (v2, hash: def)   →  Run B  →  avg_score = 3.6
                                              ↑ drop from harder questions, not a worse model
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai pandas --quiet

In [ ]:
import os
import pandas as pd
from google import genai
from google.genai import types
import mlflow

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("05-MLflow-Datasets")

print("MLflow", mlflow.__version__, "ready")

## Dataset V1 — Create, wrap and log

In [ ]:
# Version 1: 3 easy questions
df_v1 = pd.DataFrame({
    "question":     ["What is MLflow?", "What is experiment tracking?", "What is a model registry?"],
    "ground_truth": [
        "MLflow is an open-source platform for managing the ML lifecycle.",
        "Experiment tracking records runs, params and metrics for reproducibility.",
        "A model registry is a central store for versioning ML models."
    ]
})

# Wrap with mlflow.data — MLflow computes a hash automatically
dataset_v1 = mlflow.data.from_pandas(df_v1, name="eval-questions", targets="ground_truth")

print(f"Name  : {dataset_v1.name}")
print(f"Digest: {dataset_v1.digest}")

In [ ]:
# Helper: score predictions by word overlap with ground truth
def word_overlap(pred, gt):
    return len(set(gt.lower().split()) & set(pred.lower().split())) / len(set(gt.lower().split()))

# --- Run with V1 ---
with mlflow.start_run(run_name="run-dataset-v1"):
    mlflow.log_input(dataset_v1, context="eval")  # ← this links the dataset to the run
    mlflow.log_param("dataset_version", "v1")

    scores = []
    for _, row in df_v1.iterrows():
        pred = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[row["question"]],
            config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
        ).text.strip()
        scores.append(word_overlap(pred, row["ground_truth"]))

    mlflow.log_metric("avg_word_overlap", round(sum(scores)/len(scores), 4))
    print(f"V1 — avg word overlap: {sum(scores)/len(scores):.3f}")

## Dataset V2 — Add harder questions → different hash

In [ ]:
# Version 2: same 3 + 2 harder questions
df_v2 = pd.concat([df_v1, pd.DataFrame({
    "question":     ["How does MLflow tracing differ from logging?",
                     "What is the MLflow Model Registry's approval workflow?"],
    "ground_truth": [
        "Tracing captures the full execution tree of an LLM call; logging records flat key-value metrics.",
        "The registry supports Staging and Production stages with manual promotion requiring approvals."
    ]
})], ignore_index=True)

# Same name — hash will differ because data changed
dataset_v2 = mlflow.data.from_pandas(df_v2, name="eval-questions", targets="ground_truth")

print(f"V1 digest: {dataset_v1.digest}")
print(f"V2 digest: {dataset_v2.digest}  ← different hash = new version")

# --- Run with V2 ---
with mlflow.start_run(run_name="run-dataset-v2"):
    mlflow.log_input(dataset_v2, context="eval")
    mlflow.log_param("dataset_version", "v2")

    scores = []
    for _, row in df_v2.iterrows():
        pred = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[row["question"]],
            config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
        ).text.strip()
        scores.append(word_overlap(pred, row["ground_truth"]))

    mlflow.log_metric("avg_word_overlap", round(sum(scores)/len(scores), 4))
    print(f"V2 — avg word overlap: {sum(scores)/len(scores):.3f}  (lower because harder questions)")

## Log a train + eval dataset to the same run

In [ ]:
# Few-shot examples used to guide the model ("training" set)
df_train = pd.DataFrame({
    "question": ["What is a DataFrame?"],
    "answer":   ["A DataFrame is a 2D tabular data structure in pandas with rows and columns."]
})
dataset_train = mlflow.data.from_pandas(df_train, name="few-shot-examples")

with mlflow.start_run(run_name="few-shot-train-eval"):
    mlflow.log_input(dataset_train, context="train")   # few-shot examples
    mlflow.log_input(dataset_v1,   context="eval")     # evaluation questions
    mlflow.log_param("strategy", "few-shot")
    mlflow.log_metric("num_few_shot_examples", len(df_train))

    print("Run has 2 inputs: train + eval")
    print("Open this run → Overview tab → scroll down to 'Inputs' section")

## MLflow UI — What to explore
```
Datasets tab
├── eval-questions (2 versions: v1 digest, v2 digest)
│   ├── run-dataset-v1  →  avg_word_overlap (higher)
│   └── run-dataset-v2  →  avg_word_overlap (lower — harder questions)
└── few-shot-examples

Run: few-shot-train-eval → Overview → Inputs → see train + eval datasets
```

---

## All 5 notebooks complete!

| Notebook | Feature | UI Tab |
|---|---|---|
| 01 | Experiments, runs, params, metrics, artifacts | Overview |
| 02 | Tracing Gemini calls, spans | Traces |
| 03 | Chat session grouping | Traces → Sessions |
| 04 | LLM-as-a-Judge with Gemini Pro | Evaluation / Judges |
| 05 | Dataset versioning and lineage | Datasets |